In [1]:
import torch
import numpy as np
import pandas as pd
from haversine import haversine, Unit
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder, StandardScaler
from graphrfi_subgraphs import *


/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 100

# 1. Load Dataset

In [3]:
trainpath = f'../../../data/top30groups/LongLatCombined/train1/train{partition}.csv'
testpath = f'../../../data/top30groups/LongLatCombined/test1/test{partition}.csv'
traindata = pd.read_csv(trainpath, encoding='ISO-8859-1')
testdata = pd.read_csv(testpath, encoding='ISO-8859-1')

In [4]:
traindata_list, testdata_list, y_gcn, y_nrf, nrf_input, index_to_label = build_graph_data(traindata, testdata, 'weaptype1')

Feature Matrix shape:  (1790, 2)


In [5]:
from itertools import product

param_grid = {
    'embed_dim': [16, 32],
    'lr': [0.001],
    'n_tree': [40, 80],
    'tree_depth': [10],
    'feat_dropout': [0, 0.1],
    'tree_feature_rate': [0.3, 0.5],
    'batch_size': [256]
}

In [6]:
import copy

best_acc = -1
best_params = None
best_epoch = -1
results = []

# Create all combinations of the parameter grid
keys, values = zip(*param_grid.items())
i = 1
total_combinations = len(list(product(*values)))

for v in product(*values):
    # Build argument dict
    print(f"{i}/{total_combinations}")
    params = dict(zip(keys, v))
    
    # Merge with fixed defaults
    args = {
        'partition': f"gtd{partition}",
        'epochs': 1500,
        'n_class': 30,
        **params  # override with params from grid
    }

    print(f"\nRunning: {args}")

    try:
        acc, epoch, *_ = train_joint_subgraph(
            traindata_list,
            testdata_list,
            y_gcn,
            y_nrf,
            nrf_input,
            args,
            index_to_label,
            verbose=False
        )

        results.append((acc, copy.deepcopy(args)))

        if acc > best_acc:
            best_acc = acc
            best_epoch = epoch
            best_params = copy.deepcopy(args)

    except Exception as e:
        print(f"Error with params {params}: {e}")

    i = i + 1

print("\nBest Accuracy:", best_acc)
print("\nBest Epoch:", best_epoch)
print("Best Parameters:")
for k, v in best_params.items():
    print(f"{k}: {v}")


1/16

Running: {'partition': 'gtd100', 'epochs': 1500, 'n_class': 30, 'embed_dim': 16, 'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'feat_dropout': 0, 'tree_feature_rate': 0.3, 'batch_size': 256}
Best acc/epoch: 0.7756 at epoch 1389
2/16

Running: {'partition': 'gtd100', 'epochs': 1500, 'n_class': 30, 'embed_dim': 16, 'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'feat_dropout': 0, 'tree_feature_rate': 0.5, 'batch_size': 256}
Best acc/epoch: 0.7878 at epoch 1431
3/16

Running: {'partition': 'gtd100', 'epochs': 1500, 'n_class': 30, 'embed_dim': 16, 'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'feat_dropout': 0.1, 'tree_feature_rate': 0.3, 'batch_size': 256}
Best acc/epoch: 0.7922 at epoch 1277
4/16

Running: {'partition': 'gtd100', 'epochs': 1500, 'n_class': 30, 'embed_dim': 16, 'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'feat_dropout': 0.1, 'tree_feature_rate': 0.5, 'batch_size': 256}
Best acc/epoch: 0.7378 at epoch 1411
5/16

Running: {'partition': 'gtd100', 'epochs': 1500, 'n_class'

In [7]:
args = {
    'partition': f"gtd{partition}",
    'epochs': 3000,
    'n_class': 30,
    **best_params  # override with params from grid
}

acc, epoch, *_ = train_joint_subgraph(
        traindata_list,
        testdata_list,
        y_gcn,
        y_nrf,
        nrf_input,
        args,
        index_to_label,
        verbose=True
    )

print(acc, epoch)

Epoch 01 | Joint Loss: 39.1172 | NRF Acc: 0.1578
Epoch 02 | Joint Loss: 37.4171 | NRF Acc: 0.1556
Epoch 51 | Joint Loss: 3.7916 | NRF Acc: 0.4278
Epoch 101 | Joint Loss: 3.4887 | NRF Acc: 0.4556
Epoch 151 | Joint Loss: 3.1428 | NRF Acc: 0.5722
Epoch 201 | Joint Loss: 2.9075 | NRF Acc: 0.6644
Epoch 251 | Joint Loss: 2.7207 | NRF Acc: 0.6811
Epoch 301 | Joint Loss: 2.6738 | NRF Acc: 0.6544
Epoch 351 | Joint Loss: 2.5314 | NRF Acc: 0.7156
Epoch 401 | Joint Loss: 2.4219 | NRF Acc: 0.7289
Epoch 451 | Joint Loss: 2.3030 | NRF Acc: 0.7311
Epoch 501 | Joint Loss: 2.2835 | NRF Acc: 0.7256
Epoch 551 | Joint Loss: 2.2259 | NRF Acc: 0.7578
Epoch 601 | Joint Loss: 2.1932 | NRF Acc: 0.7333
Epoch 651 | Joint Loss: 2.2055 | NRF Acc: 0.7711
Epoch 701 | Joint Loss: 2.1403 | NRF Acc: 0.7811
Epoch 751 | Joint Loss: 2.1139 | NRF Acc: 0.7756
Epoch 801 | Joint Loss: 2.1522 | NRF Acc: 0.7767
Epoch 851 | Joint Loss: 2.0572 | NRF Acc: 0.7856
Epoch 901 | Joint Loss: 2.0200 | NRF Acc: 0.7889
Epoch 951 | Joint Los

In [8]:
#best_acc, best_epoch, precision, recall, f1, y_pred_decoded, y_true_decoded, precision_micro, recall_micro, f1_micro,precision_macro, recall_macro, f1_macro,roc_auc_weighted, roc_auc_micro, roc_auc_macro,epoch_logs = train_joint_subgraph(traindata_list, testdata_list, y_gcn, y_nrf, nrf_input, default_args, index_to_label, verbose=True)

In [9]:
"""
default_args = {
    'partition': f"gtd{partition}",
    'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1500,
    'feat_dropout': 0,
    'n_tree': 80,
    'tree_depth': 10,
    'tree_feature_rate': 0.5,
    'n_class': 30,
    'batch_size': 256
}
"""
#Best acc/epoch: 0.8333 at epoch 1454


'\ndefault_args = {\n    \'partition\': f"gtd{partition}",\n    \'embed_dim\': 16,\n    \'lr\': 0.001,\n    \'epochs\': 1500,\n    \'feat_dropout\': 0,\n    \'n_tree\': 80,\n    \'tree_depth\': 10,\n    \'tree_feature_rate\': 0.5,\n    \'n_class\': 30,\n    \'batch_size\': 256\n}\n'

In [10]:
#Best acc/epoch: 0.7856 at epoch 400
#Best acc/epoch: 0.8133 at epoch 1378


In [11]:
best_acc

0.8355555534362793